# This notebook is deprecated and does NOT work!
Instead, use the NNLS notebooks. It's mainly here for achieval purposes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# Set a nice default style
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['font.size'] = 12

In [ ]:
df = pd.read_excel('../../../data/spectral_library_clean.xlsx')
df.ffill(axis=0, inplace=True)
df.bfill(axis=0, inplace=True)  # handles leading NaNs

wavelength = df['Wavelength']
abs_spectra = df.iloc[:, 1:].to_numpy().T

In [ ]:
df_mix_spec = pd.read_excel("Spectral_library_with_mixtures.xlsx")
df_mix_spec.ffill(axis=0, inplace=True)
df_mix_spec.bfill(axis=0, inplace=True)

N_species = abs_spectra.shape[0]
N_mix = df_mix_spec.shape[1]

df_mix_weights = pd.read_excel("Spectral_library_mixture_weights.xlsx")

mixture_cols = df_mix_weights["mixture_name"].tolist()
mix_spectra = df_mix_spec[mixture_cols].to_numpy().T

In [ ]:
species_cols = list(df_mix_weights.columns[1:])
mixture_cols = df_mix_weights["mixture_name"].tolist()
mix_spectra = df_mix_spec[mixture_cols].to_numpy().T

df_mixtures = pd.DataFrame(
    data=mix_spectra.T,
    columns=mixture_cols
)

wavelength_mix = df_mix_spec["Wavelength"].to_numpy()
df_mixtures.insert(0, "Wavelength", wavelength_mix)

In [ ]:
X = df_mix_spec[mixture_cols].to_numpy().T
Y = df_mix_weights[species_cols].to_numpy()

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.9, random_state=0)

noise_level = 0.01

X_train_noisy = X_train + np.random.normal(0, noise_level, X_train.shape)
Y_train_noisy = Y_train + np.random.normal(0, noise_level, Y_train.shape)
X_test_noisy = X_test + np.random.normal(0, noise_level, X_test.shape)

In [ ]:
pls = PLSRegression(n_components=N_species)

pls.fit(X_train_noisy, Y_train)

Y_pred_raw = pls.predict(X_test_noisy)

Y_pred = np.clip(Y_pred_raw, 0, None)
row_sums = Y_pred.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1.0
Y_pred_clean = Y_pred / row_sums

mae = mean_absolute_error(Y_test, Y_pred_clean)
print(f"Overall Mean Absolute Error: {mae:.4f}")

for i in range(len(Y_test)):
    print(f"Test Mixture {i+1}:")
    print(f"  True : {np.round(Y_test[i], 3)}")
    print(f"  PLS  : {np.round(Y_pred_clean[i], 3)}")

In [ ]:
pred_all = np.argsort(Y_test, axis = 1)
actual_all = np.argsort(Y_pred_clean, axis = 1)
print(pred_all, actual_all)
# checks if whole list is in correct order

accuracy = np.mean(pred_all == actual_all) * 100
print(accuracy)
print(f'Random guess:{100/9}')